# Word2Vec embeddings from scratch

Лабораторная работа по построению эмбеддингов слов без готовой реализации `gensim`/`word2vec`.

В ноутбуке реализован вариант **Skip-Gram with Negative Sampling**:
- очистка русскоязычного корпуса;
- построение словаря;
- генерация положительного контекста из слов слева и справа от целевого слова;
- генерация отрицательных примеров;
- ручное обучение двух матриц эмбеддингов `W` и `C` градиентным спуском;
- сохранение словаря и эмбеддингов;
- поиск ближайших слов и проверка косинусной близости.

> Данные корпуса не добавляются в репозиторий: положите тексты в папку `data/` локально или измените `BASE_PATH` в первой ячейке.


## 1. Конфигурация

В этой ячейке задаются пути к корпусу и папке для результатов, фиксируется `seed` и задаются гиперпараметры обучения.

Ключевые параметры:
- `WINDOW = 3` — положительный контекст: 3 слова слева и 3 справа;
- `NEG_K = 10` — число отрицательных примеров;
- `EMB_DIM` — размерность вектора слова;
- `EPOCHS`, `LR` — количество эпох и шаг градиентного спуска.


In [ ]:
# Импорты и основные настройки

import os
import re
import json
import numpy as np
from pathlib import Path
from collections import Counter

# Ноутбук можно запускать и в Google Colab, и локально.
# В Colab результаты удобно сохранять на Google Drive.
try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    drive = None
    IN_COLAB = False

if IN_COLAB:
    drive.mount('/content/drive')

SEED = 42
rng_global = np.random.default_rng(SEED)

# Настройки корпуса и обучения.
# Ожидаются файлы 1.txt, 2.txt, 3.txt, 4.txt с крупным русскоязычным текстом.
BOOKS = [str(i) for i in range(1, 5)]

# Для локального запуска можно переопределить пути переменными окружения:
# WORD2VEC_BASE_PATH=data WORD2VEC_SAVE_DIR=outputs
if IN_COLAB:
    BASE_PATH = "/content/drive/MyDrive/LW2/data"
    SAVE_DIR = Path('/content/drive/MyDrive/PSRSII_embeddings_LR1')
else:
    BASE_PATH = os.getenv("WORD2VEC_BASE_PATH", "data")
    SAVE_DIR = Path(os.getenv("WORD2VEC_SAVE_DIR", "outputs"))

SAVE_DIR.mkdir(parents=True, exist_ok=True)

MIN_FREQ = 1

# В задании предлагается сравнить размеры 100, 500 и 1000.
# Здесь размер вынесен в одну константу: для повторного запуска достаточно поменять EMB_DIM.
EMB_DIM = 300

# Контекст: 3 слова слева и 3 слова справа.
WINDOW = 3

# Количество отрицательных примеров для каждого target word.
NEG_K = 10

EPOCHS = 5
LR = 0.005

print('Colab:', IN_COLAB)
print('Папка корпуса:', BASE_PATH)
print('Папка сохранения:', SAVE_DIR)


Mounted at /content/drive
Папка сохранения: /content/drive/MyDrive/PSRSII_embeddings_LR1


## 2. Чтение и предобработка корпуса

Текст приводится к нижнему регистру, знаки препинания заменяются пробелами, слова длиной 2 символа и меньше удаляются.

Важно: пунктуация именно заменяется на пробел, чтобы слова из разных частей предложения не склеивались.


In [ ]:
# Чтение и предобработка корпуса

def getParsedText(path: str) -> str:
    """Чтение и очистка текста по требованиям ЛР1.

    Пунктуацию заменяем на пробел, а не удаляем.
    Иначе "каким-нибудь" превращается в "какимнибудь",
    а слова вокруг точки/тире могут склеиваться.
    """
    with open(path, 'r', encoding='cp1251', errors='ignore') as file:
        text = file.read().lower()

    text = re.sub(r'[^а-яё\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()

    words = [w for w in text.split() if len(w) > 2]
    return ' '.join(words)


def readBook(books: list[str], base_path: str = BASE_PATH) -> str:
    parts = []
    for name in books:
        parts.append(getParsedText(os.path.join(base_path, f'{name}.txt')))
    return ' '.join(parts)


## 3. Обучение Word2Vec

Используется логика Skip-Gram:
- `w_idx` — индекс целевого слова;
- `pos` — индексы слов из реального контекста;
- `neg` — случайные слова из корпуса, которые считаются отрицательными примерами.

Для положительного контекста модель увеличивает вероятность совместной встречаемости `w` и `c_pos`, а для отрицательного контекста уменьшает вероятность совместной встречаемости `w` и `c_neg`.


In [ ]:
# Функции обучения word2vec

def sigmoid(x):
    """Стабильная сигмоида для scalar и numpy array."""
    x = np.clip(x, -30, 30)
    return 1 / (1 + np.exp(-x))


def makeEmbedding(vocab_size: int, emb_dim: int, rng=None):
    """Инициализация эмбеддингов с управляемым random seed."""
    if rng is None:
        rng = np.random.default_rng()
    limit = 1 / np.sqrt(emb_dim)
    return rng.uniform(-limit, limit, (vocab_size, emb_dim)).astype(np.float32)


def generate_pairs_ids(ids: np.ndarray, window: int = 3, neg_k: int = 10, rng=None, positions=None):
    """Генерация обучающих примеров.

    - c_pos: до 3 слов слева и 3 справа;
    - c_neg: 10 слов на target;
    - отрицательные слова выбираются из корпуса, значит с учётом частот.
    """
    if rng is None:
        rng = np.random.default_rng()

    n = len(ids)
    if positions is None:
        positions = range(n)

    for i in positions:
        w_idx = int(ids[i])

        start = max(0, i - window)
        end = min(n, i + window + 1)

        pos = [int(ids[j]) for j in range(start, end) if j != i]
        pos_set = set(pos)

        neg = []
        while len(neg) < neg_k:
            idx = int(ids[rng.integers(0, n)])
            if idx != w_idx and idx not in pos_set:
                neg.append(idx)

        yield w_idx, pos, neg


def train(text, vocab, W, C, window=3, neg_k=10, epochs=10, lr=0.005, seed=42, shuffle=True):
    """Обучение skip-gram negative sampling.

    positive: grad = (sigmoid(c*w) - 1)
    negative: grad = sigmoid(c*w)
    """
    rng = np.random.default_rng(seed)
    ids = np.array([vocab[w] for w in text], dtype=np.int32)
    n = len(ids)

    history = []

    for epoch in range(epochs):
        positions = np.arange(n)
        if shuffle:
            rng.shuffle(positions)

        total_loss = 0.0
        num_terms = 0

        for w_idx, pos, neg in generate_pairs_ids(ids, window=window, neg_k=neg_k, rng=rng, positions=positions):
            w_vec = W[w_idx].copy()
            grad_w = np.zeros_like(w_vec)

            # Положительный контекст: реальные соседние слова должны иметь высокую вероятность.
            for c_idx in pos:
                c_vec = C[c_idx].copy()
                score = float(c_vec @ w_vec)
                s = sigmoid(score)

                total_loss += -np.log(s + 1e-10)
                num_terms += 1

                coeff = s - 1
                grad_w += coeff * c_vec
                C[c_idx] -= lr * coeff * w_vec

            # Отрицательный контекст: случайные слова должны иметь низкую вероятность.
            for c_idx in neg:
                c_vec = C[c_idx].copy()
                score = float(c_vec @ w_vec)
                s = sigmoid(score)

                total_loss += -np.log(1 - s + 1e-10)
                num_terms += 1

                coeff = s
                grad_w += coeff * c_vec
                C[c_idx] -= lr * coeff * w_vec

            W[w_idx] -= lr * grad_w

        avg_loss = total_loss / max(num_terms, 1)
        history.append(avg_loss)

        print(
            f"Epoch {epoch + 1}/{epochs} | "
            f"Loss: {total_loss:.4f} | "
            f"Avg loss: {avg_loss:.6f}"
        )

    return W, C, history


## 4. Сохранение и загрузка результатов

После обучения сохраняются:
- `vocab.json` — отображение `слово -> индекс`;
- `index_to_word.json` — обратное отображение;
- `W_<dim>.npy` — target-эмбеддинги;
- `C_<dim>.npy` — context-эмбеддинги;
- `E_avg_<dim>.npy` — усреднение `W` и `C`;
- `history_<dim>.json` — история значения loss.


In [ ]:
# Сохранение и загрузка словаря/эмбеддингов с Google Drive

def saveVocab(vocab, save_dir: Path = SAVE_DIR):
    save_dir.mkdir(parents=True, exist_ok=True)

    with open(save_dir / 'vocab.json', 'w', encoding='utf-8') as f:
        json.dump(vocab, f, ensure_ascii=False, indent=2)

    # Список удобнее словаря: JSON не превращает int-ключи в строки.
    index_to_word_list = [None] * len(vocab)
    for word, idx in vocab.items():
        index_to_word_list[idx] = word

    with open(save_dir / 'index_to_word.json', 'w', encoding='utf-8') as f:
        json.dump(index_to_word_list, f, ensure_ascii=False, indent=2)


def saveEmbeddings(W, C, emb_dim: int, save_dir: Path = SAVE_DIR):
    save_dir.mkdir(parents=True, exist_ok=True)

    np.save(save_dir / f'W_{emb_dim}.npy', W)
    np.save(save_dir / f'C_{emb_dim}.npy', C)
    np.save(save_dir / f'E_avg_{emb_dim}.npy', (W + C) / 2)


def saveTrainingHistory(history, emb_dim: int, save_dir: Path = SAVE_DIR):
    save_dir.mkdir(parents=True, exist_ok=True)

    with open(save_dir / f'history_{emb_dim}.json', 'w', encoding='utf-8') as f:
        json.dump([float(x) for x in history], f, ensure_ascii=False, indent=2)


def loadVocab(save_dir: Path = SAVE_DIR):
    with open(save_dir / 'vocab.json', 'r', encoding='utf-8') as f:
        vocab = json.load(f)

    index_path = save_dir / 'index_to_word.json'
    if index_path.exists():
        with open(index_path, 'r', encoding='utf-8') as f:
            index_to_word_list = json.load(f)
        index_to_word = {i: w for i, w in enumerate(index_to_word_list)}
    else:
        index_to_word = {int(i): w for w, i in vocab.items()}

    return vocab, index_to_word


def loadEmbeddings(emb_dim: int, save_dir: Path = SAVE_DIR):
    W = np.load(save_dir / f'W_{emb_dim}.npy')
    C = np.load(save_dir / f'C_{emb_dim}.npy')
    E_avg = np.load(save_dir / f'E_avg_{emb_dim}.npy')
    return W, C, E_avg


## 5. Метрики качества и de-embedding

Для проверки реализованы:
- `nearestWord` — поиск ближайшего слова перебором через MSE или BCE;
- `nearestWordsCos` — дополнительный поиск ближайших слов по косинусной близости;
- `cos` — косинусная близость между двумя словами.

Косинус близок к 1 для похожих слов и ближе к 0 для слабо связанных слов.


In [ ]:
# Метрики и проверка ближайших слов

def MSE(word_1, word_2):
    return np.mean((word_1 - word_2) ** 2)


def binaryCrossEntropy(word_1, word_2):
    p = sigmoid(word_1)
    q = sigmoid(word_2)
    return -np.mean(p * np.log(q + 1e-10) + (1 - p) * np.log(1 - q + 1e-10))


def cos(word_idx_1, word_idx_2, E):
    denom = np.linalg.norm(E[word_idx_1]) * np.linalg.norm(E[word_idx_2])
    if denom < 1e-12:
        return 0.0
    return float((E[word_idx_1] @ E[word_idx_2]) / denom)


def nearestWord(w_idx, E, index_to_word, diff_func_type='MSE', index_skip=None):
    """Ближайшее слово перебором через MSE/BCE для требования ЛР1."""
    if index_skip is None:
        index_skip = set()
    else:
        index_skip = set(index_skip)

    min_dif = float('inf')
    res = -1

    for i in range(E.shape[0]):
        if i == w_idx or i in index_skip:
            continue

        if diff_func_type == 'MSE':
            cur_dif = MSE(E[w_idx], E[i])
        else:
            cur_dif = binaryCrossEntropy(E[w_idx], E[i])

        if cur_dif < min_dif:
            min_dif = cur_dif
            res = i

    return min_dif, index_to_word[res]


def nearestWordsCos(word, E, vocab, index_to_word, top_k=10):
    """Дополнительная диагностика: cosine обычно лучше показывает качество эмбеддингов."""
    if word not in vocab:
        return []

    idx = vocab[word]
    norms = np.linalg.norm(E, axis=1)
    denom = norms * (norms[idx] + 1e-12)
    sims = (E @ E[idx]) / (denom + 1e-12)
    sims[idx] = -np.inf

    best = np.argsort(sims)[-top_k:][::-1]
    return [(index_to_word[int(i)], float(sims[i])) for i in best]


def embedding_stats(E, name='E'):
    norms = np.linalg.norm(E, axis=1)
    print(name)
    print('finite:', np.isfinite(E).all())
    print('norm mean/std/min/max:', norms.mean(), norms.std(), norms.min(), norms.max())
    print('zero vectors:', int(np.sum(norms < 1e-8)))


## 6. Запуск обучения

Ячейка строит корпус, словарь, обучает матрицы `W` и `C`, затем сохраняет результаты.

Если корпус большой, обучение может занимать заметное время. Для отладки можно уменьшить список `BOOKS`, `EPOCHS` или размер словаря через `MIN_FREQ`.


In [ ]:
# Обучение ЛР1 и сохранение на Google Drive

text = readBook(BOOKS, BASE_PATH).split()

counter = Counter(text)
words = [w for w, freq in counter.most_common() if freq >= MIN_FREQ]

vocab = {w: i for i, w in enumerate(words)}
index_to_word = {i: w for w, i in vocab.items()}

print(f'Длина текста:   {len(text)}')
print(f'Всего уникальных слов до фильтра: {len(counter)}')
print(f'Размер словаря: {len(vocab)}')
print(f'Слов с частотой 1: {sum(1 for _, c in counter.items() if c == 1)}')

V = len(vocab)

W = makeEmbedding(V, EMB_DIM, rng_global)
C = makeEmbedding(V, EMB_DIM, rng_global)

W, C, history = train(
    text=text,
    vocab=vocab,
    W=W,
    C=C,
    window=WINDOW,
    neg_k=NEG_K,
    epochs=EPOCHS,
    lr=LR,
    seed=SEED,
    shuffle=True
)

saveVocab(vocab)
saveEmbeddings(W, C, EMB_DIM)
saveTrainingHistory(history, EMB_DIM)

print('Готово. Файлы сохранены в:', SAVE_DIR)
print('W:', SAVE_DIR / f'W_{EMB_DIM}.npy')
print('C:', SAVE_DIR / f'C_{EMB_DIM}.npy')
print('E_avg:', SAVE_DIR / f'E_avg_{EMB_DIM}.npy')
print('vocab:', SAVE_DIR / 'vocab.json')


Длина текста:   526917
Всего уникальных слов до фильтра: 62043
Размер словаря: 62043
Слов с частотой 1: 31754
Epoch 1/5 | Loss: 5810824.1100 | Avg loss: 0.689249
Epoch 2/5 | Loss: 5719443.2286 | Avg loss: 0.678410
Epoch 3/5 | Loss: 5644469.6840 | Avg loss: 0.669517
Epoch 4/5 | Loss: 5579280.1859 | Avg loss: 0.661785
Epoch 5/5 | Loss: 5519167.9012 | Avg loss: 0.654654
Готово. Файлы сохранены в: /content/drive/MyDrive/PSRSII_embeddings_LR1
W: /content/drive/MyDrive/PSRSII_embeddings_LR1/W_300.npy
C: /content/drive/MyDrive/PSRSII_embeddings_LR1/C_300.npy
E_avg: /content/drive/MyDrive/PSRSII_embeddings_LR1/E_avg_300.npy
vocab: /content/drive/MyDrive/PSRSII_embeddings_LR1/vocab.json


## 7. Проверка загрузки

Отдельная ячейка нужна, чтобы убедиться, что результаты действительно сохранены и могут быть загружены в новом запуске ноутбука.


In [ ]:
# Проверка загрузки эмбеддингов с Google Drive

vocab, index_to_word = loadVocab()
W, C, E_avg = loadEmbeddings(EMB_DIM)

print('Размер словаря:', len(vocab))
print('Форма W:', W.shape)
print('Форма C:', C.shape)
print('Форма E_avg:', E_avg.shape)


Размер словаря: 62043
Форма W: (62043, 300)
Форма C: (62043, 300)
Форма E_avg: (62043, 300)


## 8. Анализ качества эмбеддингов

Проверяются:
- статистика норм векторов;
- косинусная близость для выбранных пар слов;
- ближайшие слова по cosine;
- ближайшие слова по MSE как вариант de-embedding из задания.


In [ ]:
# Проверка качества ЛР1

embedding_stats(W, 'W')
embedding_stats(C, 'C')
embedding_stats(E_avg, 'E_avg = (W + C) / 2')

print()

test_pairs = [
    ('бог', 'бога'),
    ('стол', 'родион'),
    ('стол', 'каким'),
]

for a, b in test_pairs:
    if a in vocab and b in vocab:
        print(f'cos({a}, {b}) W:', cos(vocab[a], vocab[b], W))
    else:
        print(f'Пара пропущена, нет в словаре: {a}, {b}')

print()

for E_name, E in [('W', W), ('C', C), ('E_avg', E_avg)]:
    print('===', E_name, 'cosine nearest ===')
    for word in ['бог', 'преступление', 'наказание', 'грех', 'душа',
                 'страдание', 'любовь', 'ненависть', 'стыд', 'гордость',
                 'раскольников', 'соня', 'настасья', 'алеша',
                 'деньги', 'бедность', 'власть', 'народ', 'закон',
                 'родион', 'порфирий', 'митя', 'сонечка']:
        if word in vocab:
            print(f'{word}: {nearestWordsCos(word, E, vocab, index_to_word, top_k=5)}')
        else:
            print(f'{word}: нет в словаре')

print('=== MSE nearest для требования ЛР1, матрица W ===')
for word in ['преступление', 'наказание', 'грех', 'душа', 'бог',
             'раскольников', 'соня', 'родион', 'порфирий']:
    if word in vocab:
        _, nearest = nearestWord(vocab[word], W, index_to_word, diff_func_type='MSE')
        print(f'{word} -> {nearest}')


W
finite: True
norm mean/std/min/max: 0.610621 0.14292224 0.50699073 4.4114733
zero vectors: 0
C
finite: True
norm mean/std/min/max: 0.6108967 0.14112563 0.52072203 4.9373927
zero vectors: 0
E_avg = (W + C) / 2
finite: True
norm mean/std/min/max: 0.43152118 0.11473232 0.3530281 4.058301
zero vectors: 0

cos(бог, бога) W: 0.5624396204948425
cos(стол, родион) W: 0.13121213018894196
cos(стол, каким) W: 0.29409340023994446

=== W cosine nearest ===
бог: [('знает', 0.7118919491767883), ('черт', 0.6639924645423889), ('говорит', 0.6482242941856384), ('говорю', 0.6135085225105286), ('знаешь', 0.6054342985153198)]
преступление: [('оказалось', 0.5643662810325623), ('душе', 0.5433088541030884), ('глазах', 0.5284366011619568), ('уме', 0.5167616009712219), ('подсудимого', 0.5162880420684814)]
наказание: [('шар', 0.2730232775211334), ('выздоравливай', 0.24369806051254272), ('приступил', 0.24123524129390717), ('кириллович', 0.23787766695022583), ('свидание', 0.23676854372024536)]
грех: [('народ', 0.5